# 30D Pairwise Swiss-roll Gaussian likelihood SMC — PyTorch/ROCm Tuolumne version

This notebook is a PyTorch-only conversion of the JAX/Flax Swiss-roll Gaussian-likelihood notebook. It is intended for the **PyTorch ROCm Tuolumne** Jupyter kernel you created in Orbit.

Main changes:

- No JAX, Flax, Optax, or CUDA/NVIDIA package dependencies.
- Uses PyTorch tensors for score training, fixed-noise multiscale dynamics, and SMC propagation.
- Uses NumPy/Matplotlib for diagnostics.
- Works on ROCm/MI300A through PyTorch. PyTorch still exposes AMD GPUs through the `torch.cuda` API; this is normal on ROCm.
- Keeps the same notebook-style dynamics as the uploaded notebook: base noise defaults to `+ omega0`, while higher-level Haar noises use the `sqrt(2/t)` factor in the RHS, with no exact per-step integration.


## 0. Environment and configuration

Run this in Orbit/Jupyter with the kernel:

```text
PyTorch ROCm Tuolumne
```

The first cell should report an AMD Instinct MI300A device when running on a Tuolumne compute-node session.


In [ ]:
# ============================================================
# Imports and configuration: PyTorch/ROCm version
# ============================================================

import os
import gc
import math
import random as pyrandom
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import trange

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 0
np.random.seed(SEED)
pyrandom.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch version:", torch.__version__)
print("Torch HIP/ROCm version:", getattr(torch.version, "hip", None))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))

# PyTorch uses the CUDA namespace even for ROCm/HIP devices.
torch.set_default_dtype(torch.float32)
torch.set_num_threads(8)

# ------------------------------------------------------------
# Prior dimension
# ------------------------------------------------------------

DIM = 30

# ------------------------------------------------------------
# Pairwise Swiss-roll prior
# Every ambient coordinate is a different projection of the same
# 2D latent spiral/ribbon. There are no independent nuisance dimensions.
# ------------------------------------------------------------

SWISS_THETA_MIN = 1.5 * np.pi
SWISS_THETA_MAX = 4.5 * np.pi
SWISS_RADIUS_SCALE = 1.0
SWISS_RIBBON_STD = 0.08
SWISS_AMBIENT_NOISE_STD = 0.03
NORMALIZE_SWISS_PRIOR = True
SWISS_NORM_SAMPLES = 200_000
SWISS_NORM_SEED = 2027
SWISS_PROJ_SEED = 123

# ------------------------------------------------------------
# Score training
# ------------------------------------------------------------

TRAIN_BS = 5000
NUM_ITERATIONS = 20_000
RUN_TRAINING = True
LEARNING_RATE = 2e-4
GRAD_CLIP = 1.0

# ------------------------------------------------------------
# Lifted-grid CNN score model
# ------------------------------------------------------------

CNN_GRID_H = 4
CNN_GRID_W = 4
CNN_HIDDEN = 64
CNN_BLOCKS = 4
CNN_DENSE_HIDDEN = 128

# ------------------------------------------------------------
# Notebook-style fixed-noise multiscale SDE
# ------------------------------------------------------------

T0 = 0.01
T1 = 1.0
N_STEPS = 256

# Keep the notebook-style dynamics by default: base noise enters as +omega0.
# Use "sqrt_2_over_t" only if you intentionally want the theoretical base coefficient.
BASE_NOISE_MODE = "notebook_constant"  # "notebook_constant" or "sqrt_2_over_t"

NUM_SMC_LEVELS = 5
NUM_REF_LEVELS = NUM_SMC_LEVELS + 5
B_TAIL = 5000
BS_SMC = 5000

# ------------------------------------------------------------
# Full-dimensional Gaussian likelihood
# A = I_DIM and y_obs = mu_final.
# ------------------------------------------------------------

OBS_MODE = "identity"
OBS_DIM = DIM
GAUSSIAN_TARGET_SEED = 31415
DELTA_OBS = 0.50
OBS_NOISE_VAR = float(DELTA_OBS**2)
NORMALIZE_LIKELIHOOD_BY_DIM = False

# ------------------------------------------------------------
# Reference / target construction
# ------------------------------------------------------------

B_REF_POSTERIOR = 50_000
N_REF_POSTERIOR = 5_000
B_TARGET_REF = 5_000
N_TARGET_PLOT = 5_000

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

RUN_PRIOR_SANITY_CHECK = True
NUM_PROJECTIONS_W2 = 256
MAX_POINTS_W2 = 5000
PLOT_COORDS = [0, 1, 2]

print("Configuration loaded.")
print("DIM:", DIM)
print("NUM_SMC_LEVELS:", NUM_SMC_LEVELS)
print("NUM_REF_LEVELS:", NUM_REF_LEVELS)
print("BASE_NOISE_MODE:", BASE_NOISE_MODE)
print("OBS_MODE:", OBS_MODE)
print("OBS_NOISE_VAR:", OBS_NOISE_VAR)


In [ ]:
# ============================================================
# Basic helpers
# ============================================================

def to_np(x):
    """Convert torch tensor or array-like object to NumPy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def to_torch(x, device=DEVICE, dtype=torch.float32):
    """Convert array-like object to a torch tensor on the selected device."""
    if isinstance(x, torch.Tensor):
        return x.to(device=device, dtype=dtype)
    return torch.as_tensor(x, device=device, dtype=dtype)


def make_generator(seed: int, device=DEVICE):
    """Create a torch generator on the right device."""
    gen = torch.Generator(device=device)
    gen.manual_seed(int(seed))
    return gen


def flatten_samples_np(x):
    x = to_np(x).astype(np.float64)
    return x.reshape(x.shape[0], -1)


## 1. Pairwise Swiss-roll prior

In [ ]:
# ============================================================
# Pairwise Swiss-roll prior in R^DIM
# ============================================================

def make_pairwise_projection_matrix_np(dim, seed=0):
    """
    Make B in R^{dim x 2}.

    Rows are evenly spaced directions on the unit circle. Any two
    distinct rows are non-collinear, so each coordinate pair gives a
    non-degenerate linear view of the same 2D latent spiral geometry.
    """
    angles = np.linspace(0.0, np.pi, dim, endpoint=False)
    rng = np.random.default_rng(seed)
    offset = rng.uniform(0.0, np.pi / dim)
    angles = angles + offset

    B = np.stack([np.cos(angles), np.sin(angles)], axis=1).astype(np.float32)
    signs = rng.choice([-1.0, 1.0], size=(dim, 1)).astype(np.float32)
    return B * signs


B_SWISS_np = make_pairwise_projection_matrix_np(DIM, seed=SWISS_PROJ_SEED)
B_SWISS_torch = torch.as_tensor(B_SWISS_np, device=DEVICE, dtype=torch.float32)

print("Pairwise Swiss-roll prior")
print("B_SWISS shape:", B_SWISS_np.shape)
print("first five rows of B_SWISS:\n", B_SWISS_np[:5])


def sample_pairwise_swiss_raw_np(n, seed=0):
    """Raw unnormalized NumPy samples, shape (n, DIM)."""
    rng = np.random.default_rng(seed)

    theta = rng.uniform(SWISS_THETA_MIN, SWISS_THETA_MAX, size=(n,)).astype(np.float32)
    r = SWISS_RADIUS_SCALE * theta

    z_center = np.stack([r * np.cos(theta), r * np.sin(theta)], axis=1).astype(np.float32)

    if SWISS_RIBBON_STD > 0:
        z = z_center + SWISS_RIBBON_STD * rng.normal(size=z_center.shape).astype(np.float32)
    else:
        z = z_center

    x = z @ B_SWISS_np.T

    if SWISS_AMBIENT_NOISE_STD > 0:
        x = x + SWISS_AMBIENT_NOISE_STD * rng.normal(size=x.shape).astype(np.float32)

    return x.astype(np.float32)


# Fixed coordinate normalization.
if NORMALIZE_SWISS_PRIOR:
    X_norm_np = sample_pairwise_swiss_raw_np(SWISS_NORM_SAMPLES, seed=SWISS_NORM_SEED)
    SWISS_MEAN_np = X_norm_np.mean(axis=0).astype(np.float32)
    SWISS_STD_np = (X_norm_np.std(axis=0) + 1e-6).astype(np.float32)
else:
    SWISS_MEAN_np = np.zeros((DIM,), dtype=np.float32)
    SWISS_STD_np = np.ones((DIM,), dtype=np.float32)

SWISS_MEAN_torch = torch.as_tensor(SWISS_MEAN_np, device=DEVICE, dtype=torch.float32)
SWISS_STD_torch = torch.as_tensor(SWISS_STD_np, device=DEVICE, dtype=torch.float32)


def sample_data_np(n, seed=0):
    """Direct prior samples as NumPy array, shape (n, DIM)."""
    x = sample_pairwise_swiss_raw_np(n, seed=seed)
    x = (x - SWISS_MEAN_np[None, :]) / SWISS_STD_np[None, :]
    return x.astype(np.float32)


def sample_data_torch(batch_size, seed: Optional[int] = None, device=DEVICE):
    """Direct prior samples as torch tensor, shape (batch_size, DIM)."""
    if seed is None:
        theta = torch.empty((batch_size,), device=device).uniform_(SWISS_THETA_MIN, SWISS_THETA_MAX)
        ribbon_noise = torch.randn((batch_size, 2), device=device)
        ambient_noise = torch.randn((batch_size, DIM), device=device)
    else:
        gen = make_generator(seed, device=device)
        theta = torch.empty((batch_size,), device=device).uniform_(SWISS_THETA_MIN, SWISS_THETA_MAX, generator=gen)
        ribbon_noise = torch.randn((batch_size, 2), device=device, generator=gen)
        ambient_noise = torch.randn((batch_size, DIM), device=device, generator=gen)

    r = SWISS_RADIUS_SCALE * theta
    z_center = torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)

    if SWISS_RIBBON_STD > 0:
        z = z_center + SWISS_RIBBON_STD * ribbon_noise
    else:
        z = z_center

    x = z @ B_SWISS_torch.T

    if SWISS_AMBIENT_NOISE_STD > 0:
        x = x + SWISS_AMBIENT_NOISE_STD * ambient_noise

    x = (x - SWISS_MEAN_torch[None, :]) / SWISS_STD_torch[None, :]
    return x.float()


# Backward-compatible alias for older cells.
sample_swiss_roll_np = sample_data_np

# Smoke test and pairwise visualization.
X_test_np = sample_data_np(8000, seed=0)
print("X_test_np:", X_test_np.shape)
print("mean first 5:", X_test_np.mean(axis=0)[:5])
print("std first 5:", X_test_np.std(axis=0)[:5])

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
if DIM > 5:
    pairs += [(3, 4), (4, 5)]
if DIM > 9:
    pairs += [(6, 7), (8, 9)]

n_cols = 4
n_rows = int(np.ceil(len(pairs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.6 * n_rows), squeeze=False)
axes = axes.ravel()

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(X_test_np[:, i], X_test_np[:, j], s=4, alpha=0.35)
    ax.set_xlabel(f"x{i}")
    ax.set_ylabel(f"x{j}")
    ax.set_title(f"x{i} vs x{j}")
    ax.grid(True, alpha=0.3)

for ax in axes[len(pairs):]:
    ax.axis("off")

plt.suptitle("Pairwise Swiss-roll prior projections", y=1.02)
plt.tight_layout()
plt.show()


## 2. Gaussian likelihood

In [ ]:
# ============================================================
# Pairwise Swiss-roll prior and full-dimensional Gaussian likelihood
# ============================================================

# Draw target from the current prior.
mu_final_np = sample_data_np(1, seed=GAUSSIAN_TARGET_SEED)[0]
mu_final_torch = torch.as_tensor(mu_final_np, device=DEVICE, dtype=torch.float32)

# Identity observation in R^DIM.
A_OBS_np = np.eye(DIM, dtype=np.float32)
A_OBS_torch = torch.eye(DIM, device=DEVICE, dtype=torch.float32)
y_obs_np = mu_final_np.copy()
y_obs_torch = mu_final_torch.clone()

OBS_MODE = "identity"
OBS_DIM = DIM


def apply_observation_torch(x):
    """Full-dimensional Gaussian observation: A x = x."""
    return x


def apply_observation_np(x):
    """Full-dimensional Gaussian observation in NumPy."""
    return np.asarray(x)


def log_final_gaussian_likelihood_torch(x, eps=1e-8):
    """
    Final likelihood:
        log l(x) = -1/2 ||x - mu_final||^2 / OBS_NOISE_VAR.
    """
    res = apply_observation_torch(x) - y_obs_torch[None, :]
    quad_per_coord = (res**2) / (OBS_NOISE_VAR + eps)

    if NORMALIZE_LIKELIHOOD_BY_DIM:
        quad = torch.mean(quad_per_coord, dim=1)
    else:
        quad = torch.sum(quad_per_coord, dim=1)

    return -0.5 * quad


def log_final_gaussian_likelihood_np(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float64)
    res = apply_observation_np(x) - y_obs_np[None, :]
    quad_per_coord = (res**2) / (OBS_NOISE_VAR + eps)
    quad = np.mean(quad_per_coord, axis=1) if NORMALIZE_LIKELIHOOD_BY_DIM else np.sum(quad_per_coord, axis=1)
    return -0.5 * quad

print("OBS_MODE:", OBS_MODE)
print("DIM:", DIM)
print("OBS_DIM:", OBS_DIM)
print("A_OBS shape:", A_OBS_np.shape)
print("OBS_NOISE_VAR:", OBS_NOISE_VAR)
print("NORMALIZE_LIKELIHOOD_BY_DIM:", NORMALIZE_LIKELIHOOD_BY_DIM)
print("Gaussian target mu_final:", mu_final_np)

# Prior visualization with target.
prior_demo = sample_data_np(8000, seed=123)
print("prior_demo shape:", prior_demo.shape)

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
if DIM > 5:
    pairs += [(3, 4), (4, 5)]
if DIM > 9:
    pairs += [(6, 7), (8, 9)]
if DIM > 15:
    pairs += [(10, 11), (12, 15)]

n_cols = 4
n_rows = int(np.ceil(len(pairs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.6 * n_rows), squeeze=False)
axes = axes.ravel()

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(prior_demo[:, i], prior_demo[:, j], s=4, alpha=0.35, label="prior")
    ax.scatter([mu_final_np[i]], [mu_final_np[j]], c="red", marker="x", s=120, linewidths=2, label="Gaussian target")
    ax.set_xlabel(f"x{i}")
    ax.set_ylabel(f"x{j}")
    ax.set_title(f"x{i} vs x{j}")
    ax.grid(True, alpha=0.3)

for ax in axes[len(pairs):]:
    ax.axis("off")

axes[0].legend(fontsize=8)
plt.suptitle("Pairwise Swiss-roll prior projections and Gaussian target", y=1.02)
plt.tight_layout()
plt.show()


## 3. PyTorch lifted-grid CNN score model

In [ ]:
# ============================================================
# Noising schedule and lifted-grid CNN score model in PyTorch
# ============================================================

beta_0 = 0.1
beta_1 = 20.0


def log_alpha_torch(t):
    return -0.5 * t * beta_0 - 0.25 * (t**2) * (beta_1 - beta_0)


def log_sigma_torch(t):
    return 0.5 * torch.log(torch.clamp(2.0 * t - t**2, min=1e-8))


class PeriodicConvBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(hidden, hidden, kernel_size=3, padding=0)
        self.conv2 = nn.Conv2d(hidden, hidden, kernel_size=3, padding=0)

    def forward(self, h):
        residual = h
        h = F.pad(h, (1, 1, 1, 1), mode="circular")
        h = self.conv1(h)
        h = F.silu(h)
        h = F.pad(h, (1, 1, 1, 1), mode="circular")
        h = self.conv2(h)
        return F.silu(h + residual)


class SwissRollScoreCNN(nn.Module):
    """
    CNN-style score model for DIM-dimensional embedded Swiss-roll data.

    Interface:
        model(t, x) -> score with shape (B, DIM)

    Since x is not an image, we lift vector features into a small learned
    spatial grid and then use residual periodic-convolution blocks.
    """
    def __init__(self, dim, grid_h=4, grid_w=4, hidden=64, num_blocks=4, dense_hidden=128):
        super().__init__()
        self.dim = dim
        self.grid_h = grid_h
        self.grid_w = grid_w
        self.hidden = hidden

        # Features: x (DIM), ||x||^2 (1), time features (3)
        in_features = dim + 1 + 3
        self.lift = nn.Linear(in_features, grid_h * grid_w * hidden)

        # Add two coordinate channels before the first convolution.
        self.conv_in = nn.Conv2d(hidden + 2, hidden, kernel_size=3, padding=0)
        self.blocks = nn.ModuleList([PeriodicConvBlock(hidden) for _ in range(num_blocks)])

        self.head = nn.Sequential(
            nn.Linear(hidden, dense_hidden),
            nn.SiLU(),
            nn.Linear(dense_hidden, dense_hidden),
            nn.SiLU(),
            nn.Linear(dense_hidden, dim),
        )

    def _prepare_t(self, t, batch_size, device, dtype):
        if not isinstance(t, torch.Tensor):
            t = torch.tensor(float(t), device=device, dtype=dtype)
        else:
            t = t.to(device=device, dtype=dtype)

        if t.ndim == 0:
            t = torch.ones((batch_size, 1), device=device, dtype=dtype) * t
        elif t.ndim == 1:
            t = t[:, None]
        elif t.ndim == 2:
            pass
        else:
            raise ValueError(f"Unexpected t shape: {tuple(t.shape)}")
        return t

    def forward(self, t, x):
        B = x.shape[0]
        x = x.float()
        t = self._prepare_t(t, B, x.device, x.dtype)

        time_feats = torch.cat(
            [
                t,
                torch.sin(2.0 * math.pi * t),
                torch.cos(2.0 * math.pi * t),
            ],
            dim=-1,
        )

        x_norm2 = torch.sum(x**2, dim=1, keepdim=True)
        feats = torch.cat([x, x_norm2, time_feats], dim=-1)

        h = F.silu(self.lift(feats))
        h = h.reshape(B, self.grid_h, self.grid_w, self.hidden).permute(0, 3, 1, 2).contiguous()

        yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, self.grid_h, device=x.device, dtype=x.dtype),
            torch.linspace(-1.0, 1.0, self.grid_w, device=x.device, dtype=x.dtype),
            indexing="ij",
        )
        coords = torch.stack([yy, xx], dim=0)[None, :, :, :].expand(B, -1, -1, -1)
        h = torch.cat([h, coords], dim=1)

        h = F.pad(h, (1, 1, 1, 1), mode="circular")
        h = F.silu(self.conv_in(h))

        for block in self.blocks:
            h = block(h)

        h = h.mean(dim=(2, 3))
        return self.head(h)


model = SwissRollScoreCNN(
    dim=DIM,
    grid_h=CNN_GRID_H,
    grid_w=CNN_GRID_W,
    hidden=CNN_HIDDEN,
    num_blocks=CNN_BLOCKS,
    dense_hidden=CNN_DENSE_HIDDEN,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print("CNN score model initialized.")
print("CNN grid:", (CNN_GRID_H, CNN_GRID_W))
print("CNN hidden channels:", CNN_HIDDEN)
print("CNN blocks:", CNN_BLOCKS)


In [ ]:
# ============================================================
# Denoising score matching in PyTorch
# ============================================================

def sm_loss_torch(model, bs):
    x0 = sample_data_torch(bs, seed=None, device=DEVICE)
    t = torch.empty((bs, 1), device=DEVICE).uniform_(1e-4, 1.0 - 1e-4)
    eps = torch.randn_like(x0)

    rev_t = 1.0 - t
    alpha = torch.exp(log_alpha_torch(rev_t))
    sigma = torch.exp(log_sigma_torch(rev_t))
    x_noisy = alpha * x0 + sigma * eps

    pred = model(rev_t, x_noisy)
    return torch.mean(torch.sum((eps + sigma * pred) ** 2, dim=1))


# One smoke step
model.train()
optimizer.zero_grad(set_to_none=True)
smoke_loss = sm_loss_torch(model, TRAIN_BS)
smoke_loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
optimizer.step()
print("smoke loss:", float(smoke_loss.detach().cpu()))


In [ ]:
# ============================================================
# Train score model
# ============================================================

if RUN_TRAINING:
    model.train()
    loss_plot = np.zeros(NUM_ITERATIONS, dtype=np.float32)

    for it in trange(NUM_ITERATIONS):
        optimizer.zero_grad(set_to_none=True)
        loss = sm_loss_torch(model, TRAIN_BS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        loss_plot[it] = float(loss.detach().cpu())
        if (it + 1) % 1000 == 0:
            print(f"iter {it+1}: loss={loss_plot[it]:.6f}")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    plt.figure(figsize=(6, 4))
    plt.plot(loss_plot)
    plt.yscale("log")
    plt.grid(True, alpha=0.3)
    plt.title("PyTorch CNN score matching loss")
    plt.xlabel("iteration")
    plt.ylabel("loss")
    plt.tight_layout()
    plt.show()
else:
    print("RUN_TRAINING=False; using current initialized score model.")

model.eval()


## 4. Notebook-style fixed-noise multiscale SDE simulator

In [ ]:
# ============================================================
# Notebook-style fixed-noise multiscale SDE simulator in R^DIM
# No exact per-time-step integration of the noise coefficients.
# ============================================================

@torch.no_grad()
def base_drift_torch(t, y, model):
    """v_t(y) = (y + 2 score_{1-t}(y)) / t."""
    if not isinstance(t, torch.Tensor):
        t = torch.tensor(float(t), device=y.device, dtype=y.dtype)
    t_safe = torch.clamp(t, min=1e-6)
    tau = 1.0 - t_safe
    score = model(tau, y)
    return (y + 2.0 * score) / t_safe


@torch.no_grad()
def vector_field_torch(t, x, model):
    return base_drift_torch(t, x, model)


def psi_pair_np(t, n, k):
    """Haar-like pair basis, NumPy/Python version. Returns shape (2,)."""
    d = 2.0 ** (-(n - 1))
    I0 = (k - 1) * d
    I1 = k * d
    I2 = (k + 1) * d
    scale = 2.0 ** ((n - 2) / 2)

    if (t > I0) and (t < I1):
        return np.array([1.0, 0.0], dtype=np.float32) * scale
    if (t > I1) and (t < I2):
        return np.array([0.0, -1.0], dtype=np.float32) * scale
    return np.array([0.0, 0.0], dtype=np.float32)


def white_level_coeffs_np(t, n):
    if n == 1:
        return np.ones((1,), dtype=np.float32)
    num_k_iter = 2 ** (n - 2)
    k_values = 2 * np.arange(num_k_iter) + 1
    psi_vec = np.stack([psi_pair_np(float(t), n, int(k)) for k in k_values], axis=0)
    psi_flat = psi_vec.reshape(-1)
    scal = 2.0 ** (-(n) / 2)
    return psi_flat * scal


def generate_eta_tensor_torch(seed, n_level, batch_size, dim=DIM):
    gen = make_generator(seed, device=DEVICE)
    num_coeffs = 2 ** (n_level - 1)
    return torch.randn((batch_size, num_coeffs, dim), device=DEVICE, generator=gen)


def white_level_vector_torch(t, n_level, eta_batch):
    """
    Notebook-style white-level coefficient generalized to R^DIM.
    eta_batch: (batch_size, 2**(n-1), dim)
    returns: (batch_size, dim)
    """
    t_float = float(t.detach().cpu()) if isinstance(t, torch.Tensor) else float(t)
    coeffs_np = white_level_coeffs_np(t_float, n_level)
    coeffs = torch.as_tensor(coeffs_np, device=eta_batch.device, dtype=eta_batch.dtype)

    coeff_noise = torch.sum(coeffs[None, :, None] * eta_batch, dim=1)
    t_safe = torch.clamp(t if isinstance(t, torch.Tensor) else torch.tensor(t_float, device=eta_batch.device), min=1e-6)
    return coeff_noise * torch.sqrt(2.0 / t_safe)


@torch.no_grad()
def simulate_base_level_torch(seed, batch_size, model, n_steps=N_STEPS, t0=T0, t1=T1):
    """
    Simulate base prior level on a fixed time grid.

    Default notebook RHS:
        dY_t/dt = v_t(Y_t) + omega0.
    """
    gen = make_generator(seed, device=DEVICE)
    y = torch.randn((batch_size, DIM), device=DEVICE, generator=gen)
    omega0 = torch.randn((batch_size, DIM), device=DEVICE, generator=gen)

    ts = torch.linspace(t0, t1, n_steps + 1, device=DEVICE)
    dt = (t1 - t0) / n_steps

    ys = [y]
    for k in range(n_steps):
        t = ts[k]
        drift = base_drift_torch(t, y, model)
        if BASE_NOISE_MODE == "sqrt_2_over_t":
            noise = torch.sqrt(2.0 / torch.clamp(t, min=1e-6)) * omega0
        else:
            noise = omega0
        y = y + dt * (drift + noise)
        ys.append(y)

    return torch.stack(ys, dim=0)


@torch.no_grad()
def increment_drift_torch(t, y, x_base, model):
    return base_drift_torch(t, x_base + y, model) - base_drift_torch(t, x_base, model)


@torch.no_grad()
def simulate_increment_level_torch(seed, noise_level, current_paths, model, n_steps=N_STEPS, t0=T0, t1=T1):
    batch_size = current_paths.shape[1]
    eta_batch = generate_eta_tensor_torch(seed, noise_level, batch_size, DIM)
    y = torch.zeros((batch_size, DIM), device=DEVICE, dtype=current_paths.dtype)

    ts = torch.linspace(t0, t1, n_steps + 1, device=DEVICE)
    dt = (t1 - t0) / n_steps

    ys = [y]
    for k in range(n_steps):
        t = ts[k]
        x_base = current_paths[k]
        drift = increment_drift_torch(t, y, x_base, model)
        noise = white_level_vector_torch(t, noise_level, eta_batch)
        y = y + dt * (drift + noise)
        ys.append(y)

    return torch.stack(ys, dim=0)


@torch.no_grad()
def simulate_prior_hierarchy_with_increments_torch(seed, batch_size, num_levels, model, n_steps=N_STEPS):
    paths_by_level = []
    increments_by_level = []

    current_paths = simulate_base_level_torch(seed, batch_size, model, n_steps=n_steps)
    paths_by_level.append(current_paths)
    increments_by_level.append(current_paths)

    for ell in range(1, num_levels):
        inc_paths = simulate_increment_level_torch(
            seed + 1009 * ell,
            noise_level=ell + 1,
            current_paths=current_paths,
            model=model,
            n_steps=n_steps,
        )
        current_paths = current_paths + inc_paths
        increments_by_level.append(inc_paths)
        paths_by_level.append(current_paths)

    return paths_by_level, increments_by_level


def check_increment_identity(paths_by_level, increments_by_level):
    for ell in range(1, len(paths_by_level)):
        err = torch.max(torch.abs(paths_by_level[ell] - paths_by_level[ell - 1] - increments_by_level[ell]))
        print(f"level {ell+1}: max |S_new - S_prev - inc| = {float(err.detach().cpu()):.3e}")


print("PyTorch notebook-style multiscale SDE simulator defined in R^DIM.")
print("DIM:", DIM)
print("BASE_NOISE_MODE:", BASE_NOISE_MODE)


## 5. Prior-generation sanity check

In [ ]:
# ============================================================
# Prior-generation sanity check in full DIM-dimensional space
# ============================================================

def sliced_w2_samples(X, Y, num_projections=NUM_PROJECTIONS_W2, max_points=MAX_POINTS_W2, seed=0):
    X = flatten_samples_np(X)
    Y = flatten_samples_np(Y)
    rng = np.random.default_rng(seed)

    if len(X) > max_points:
        X = X[rng.choice(len(X), size=max_points, replace=False)]
    if len(Y) > max_points:
        Y = Y[rng.choice(len(Y), size=max_points, replace=False)]

    k = min(len(X), len(Y))
    X, Y = X[:k], Y[:k]

    dirs = rng.normal(size=(num_projections, X.shape[1]))
    dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)

    Xp = np.sort(X @ dirs.T, axis=0)
    Yp = np.sort(Y @ dirs.T, axis=0)
    return np.sqrt(np.mean((Xp - Yp) ** 2))


if RUN_PRIOR_SANITY_CHECK:
    model.eval()
    paths_prior_check, _ = simulate_prior_hierarchy_with_increments_torch(
        seed=12345,
        batch_size=5000,
        num_levels=NUM_SMC_LEVELS,
        model=model,
        n_steps=N_STEPS,
    )

    direct_prior_1 = sample_data_np(5000, seed=900)
    direct_prior_2 = sample_data_np(5000, seed=901)

    floor = sliced_w2_samples(direct_prior_1, direct_prior_2, seed=10)
    gen_vs_prior = np.array([
        sliced_w2_samples(paths_prior_check[ell][-1], direct_prior_1, seed=100 + ell)
        for ell in range(NUM_SMC_LEVELS)
    ])

    levels = np.arange(1, NUM_SMC_LEVELS + 1)
    plt.figure(figsize=(6.5, 4.2))
    plt.semilogy(levels, gen_vs_prior, marker="o", label="generated level vs direct prior")
    plt.axhline(floor, linestyle="--", label=f"direct-prior floor = {floor:.4f}")
    plt.xlabel("level")
    plt.ylabel("sliced W2")
    plt.title("Prior generation sanity check")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("generated vs direct prior sliced W2:", gen_vs_prior)
    print("direct-prior finite-sample floor:", floor)
else:
    print("RUN_PRIOR_SANITY_CHECK=False; skipping prior-generation sanity check.")


## 6. Prior-gap likelihood hierarchy

In [ ]:
# ============================================================
# Observation-space likelihood hierarchy, NumPy/Torch version
# ============================================================

def empirical_diag_var_obs_np(Z):
    Z = np.asarray(Z, dtype=np.float64)
    Zc = Z - Z.mean(axis=0, keepdims=True)
    return np.sum(Zc**2, axis=0) / max(Z.shape[0] - 1, 1)


def avg_obs_var_np(Z):
    return float(np.mean(empirical_diag_var_obs_np(Z)))


def make_observation_prior_gap_likelihood_hierarchy_np(paths_by_level, obs_noise_var, num_smc_levels, ref_index=-1, use_monotone=True, eps=1e-12):
    finals = np.stack([to_np(p[-1]) for p in paths_by_level], axis=0)
    S_ref = finals[ref_index]
    Z_ref = apply_observation_np(S_ref)
    tau_ref2 = avg_obs_var_np(Z_ref)

    tail_means_obs = []
    tail_diag_vars_obs = []
    level_vars = []
    tail_vars = []
    gap_vars = []

    for ell in range(num_smc_levels):
        Z_ell = apply_observation_np(finals[ell])
        E_ell = Z_ref - Z_ell
        mu_E = E_ell.mean(axis=0)
        diag_var_E = empirical_diag_var_obs_np(E_ell)
        tau_level2 = avg_obs_var_np(Z_ell)
        tau_tail2 = float(np.mean(diag_var_E))
        gap = max(tau_ref2 - tau_level2, 0.0)

        tail_means_obs.append(mu_E)
        tail_diag_vars_obs.append(diag_var_E)
        level_vars.append(tau_level2)
        tail_vars.append(tau_tail2)
        gap_vars.append(gap)

    tail_means_obs = np.stack(tail_means_obs).astype(np.float32)
    tail_diag_vars_obs = np.stack(tail_diag_vars_obs).astype(np.float32)
    level_vars = np.asarray(level_vars, dtype=np.float64)
    tail_vars = np.asarray(tail_vars, dtype=np.float64)
    gap_vars = np.asarray(gap_vars, dtype=np.float64)

    if use_monotone:
        gap_vars = np.minimum.accumulate(gap_vars)

    eta = np.sqrt(gap_vars / (tail_vars + eps)).astype(np.float32)

    obs_biases = []
    obs_vars = []
    for ell in range(num_smc_levels):
        b_ell = eta[ell] * tail_means_obs[ell]
        var_ell = obs_noise_var + (eta[ell] ** 2) * tail_diag_vars_obs[ell]
        obs_biases.append(b_ell)
        obs_vars.append(var_ell)

    return {
        "obs_biases": np.stack(obs_biases).astype(np.float32),
        "obs_vars": np.stack(obs_vars).astype(np.float32),
        "eta": eta,
        "gap_vars": gap_vars,
        "tail_vars": tail_vars,
        "level_vars": level_vars,
        "tau_ref2": tau_ref2,
        "tail_means_obs": tail_means_obs,
        "tail_diag_vars_obs": tail_diag_vars_obs,
    }


model.eval()
paths_prior_pilot, increments_prior_pilot = simulate_prior_hierarchy_with_increments_torch(
    seed=222,
    batch_size=B_TAIL,
    num_levels=NUM_REF_LEVELS,
    model=model,
    n_steps=N_STEPS,
)
check_increment_identity(paths_prior_pilot, increments_prior_pilot)

obs_like = make_observation_prior_gap_likelihood_hierarchy_np(
    paths_prior_pilot,
    obs_noise_var=OBS_NOISE_VAR,
    num_smc_levels=NUM_SMC_LEVELS,
    ref_index=-1,
    use_monotone=True,
)

obs_biases_np = obs_like["obs_biases"]
obs_vars_np = obs_like["obs_vars"]
obs_biases_torch = torch.as_tensor(obs_biases_np, device=DEVICE, dtype=torch.float32)
obs_vars_torch = torch.as_tensor(obs_vars_np, device=DEVICE, dtype=torch.float32)

print("Observation-space likelihood hierarchy:")
print("tau_ref^2:", obs_like["tau_ref2"])
for ell in range(NUM_SMC_LEVELS):
    print(
        f"level {ell+1}: tau_level^2={obs_like['level_vars'][ell]:.6f}, "
        f"gap={obs_like['gap_vars'][ell]:.6f}, "
        f"tail_var={obs_like['tail_vars'][ell]:.6f}, "
        f"eta={obs_like['eta'][ell]:.4f}, mean obs_var={np.mean(obs_vars_np[ell]):.6f}"
    )


## 7. Reference posterior and level-wise target distributions

In [ ]:
# ============================================================
# Reference posterior and level-wise target distributions Q_n
# ============================================================

def normalize_logweights_np(logw):
    lw = np.asarray(logw, dtype=np.float64)
    lw = lw - np.max(lw)
    w = np.exp(lw)
    return w / np.sum(w)


def weighted_resample_np(X, logw, num_samples, seed=0):
    w = normalize_logweights_np(logw)
    ess = 1.0 / np.sum(w**2)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=num_samples, replace=True, p=w)
    return X[idx], w, ess, idx


def log_observation_gaussian_unnorm_np(x, y_obs, obs_bias, obs_var, normalize_by_dim=False, eps=1e-8):
    Ax = apply_observation_np(x)
    res = y_obs[None, :] - Ax - obs_bias[None, :]
    quad_per_coord = (res**2) / (obs_var[None, :] + eps)
    quad = np.mean(quad_per_coord, axis=1) if normalize_by_dim else np.sum(quad_per_coord, axis=1)
    return -0.5 * quad


# Independent target posterior from direct prior samples and the true final likelihood.
prior_samples_ref = sample_data_np(B_REF_POSTERIOR, seed=500)
logw_ref = log_final_gaussian_likelihood_np(prior_samples_ref)
posterior_samples, posterior_weights, posterior_ref_ess, _ = weighted_resample_np(
    prior_samples_ref,
    logw_ref,
    num_samples=N_REF_POSTERIOR,
    seed=501,
)

print("Reference posterior samples:", posterior_samples.shape)
print("Reference posterior ESS:", posterior_ref_ess)
print("Reference posterior ESS/N:", posterior_ref_ess / B_REF_POSTERIOR)


def build_levelwise_target_reference(seed=456, num_ref_samples=B_TARGET_REF, num_target_samples=N_TARGET_PLOT):
    """Build empirical Q_n ∝ p_n l_n using generated prior levels and level likelihoods."""
    paths_ref, _ = simulate_prior_hierarchy_with_increments_torch(
        seed=seed,
        batch_size=num_ref_samples,
        num_levels=NUM_SMC_LEVELS,
        model=model,
        n_steps=N_STEPS,
    )

    target_samples_by_level = []
    target_weights_by_level = []
    target_ess_by_level = []
    prior_level_samples = []

    for ell in range(NUM_SMC_LEVELS):
        X_ell_np = to_np(paths_ref[ell][-1])
        logw_ell = log_observation_gaussian_unnorm_np(
            X_ell_np,
            y_obs_np,
            obs_biases_np[ell],
            obs_vars_np[ell],
            NORMALIZE_LIKELIHOOD_BY_DIM,
        )
        X_target, w, ess, idx = weighted_resample_np(
            X_ell_np,
            logw_ell,
            num_samples=num_target_samples,
            seed=seed + ell + 100,
        )

        target_samples_by_level.append(X_target)
        target_weights_by_level.append(w)
        target_ess_by_level.append(ess)
        prior_level_samples.append(X_ell_np)

        print(f"target Q_{ell+1}: reference ESS={ess:.2f}, ESS/N={ess/len(X_ell_np):.4f}")

    return {
        "target_samples_by_level": target_samples_by_level,
        "target_weights_by_level": target_weights_by_level,
        "target_ess_by_level": np.asarray(target_ess_by_level),
        "prior_level_samples": prior_level_samples,
    }


target_ref = build_levelwise_target_reference(seed=456)
target_samples_by_level = target_ref["target_samples_by_level"]
target_ess_by_level = target_ref["target_ess_by_level"]


## 8. Branch/kill SMC

In [ ]:
# ============================================================
# Branch/kill SMC in PyTorch
# ============================================================

def normalize_logweights_torch(logw):
    return torch.exp(logw - torch.logsumexp(logw, dim=0))


def ess_from_logweights_torch(logw):
    w = normalize_logweights_torch(logw)
    return 1.0 / torch.sum(w**2)


def log_observation_gaussian_unnorm_torch(x, y_obs, obs_bias, obs_var, normalize_by_dim=False, eps=1e-8):
    Ax = apply_observation_torch(x)
    res = y_obs[None, :] - Ax - obs_bias[None, :]
    quad_per_coord = (res**2) / (obs_var[None, :] + eps)
    quad = torch.mean(quad_per_coord, dim=1) if normalize_by_dim else torch.sum(quad_per_coord, dim=1)
    return -0.5 * quad


def stochastic_round_resample_paths_torch(paths, logw, seed, target_N=None):
    N_current = paths.shape[1]
    if target_N is None:
        target_N = N_current

    w = normalize_logweights_torch(logw)
    gen = make_generator(seed, device=paths.device)
    U = torch.rand((N_current,), device=paths.device, generator=gen)
    counts = torch.floor(target_N * w + U).long()

    idx = torch.repeat_interleave(torch.arange(N_current, device=paths.device), counts)
    if idx.numel() == 0:
        idx = torch.argmax(w).reshape(1)

    return paths[:, idx, :], counts, idx


@torch.no_grad()
def run_multilevel_smc_observation(seed, batch_size, target_N=None):
    if target_N is None:
        target_N = batch_size

    samples_by_level = []
    paths_by_level = []
    ess_by_level = []
    pop_by_level = []
    counts_by_level = []

    current_paths = simulate_base_level_torch(seed + 1, batch_size, model, n_steps=N_STEPS)

    logw = log_observation_gaussian_unnorm_torch(
        current_paths[-1],
        y_obs_torch,
        obs_biases_torch[0],
        obs_vars_torch[0],
        NORMALIZE_LIKELIHOOD_BY_DIM,
    )
    ess = ess_from_logweights_torch(logw)
    current_paths, counts, idx = stochastic_round_resample_paths_torch(current_paths, logw, seed + 2, target_N)

    samples_by_level.append(current_paths[-1])
    paths_by_level.append(current_paths)
    ess_by_level.append(float(ess.detach().cpu()))
    pop_by_level.append(current_paths.shape[1])
    counts_by_level.append(to_np(counts))
    print(f"After level 1: N={current_paths.shape[1]}, ESS={ess_by_level[-1]:.2f}")

    for ell in range(1, NUM_SMC_LEVELS):
        inc_paths = simulate_increment_level_torch(seed + 1000 + ell, ell + 1, current_paths, model, n_steps=N_STEPS)
        proposed_paths = current_paths + inc_paths

        logw_new = log_observation_gaussian_unnorm_torch(
            proposed_paths[-1],
            y_obs_torch,
            obs_biases_torch[ell],
            obs_vars_torch[ell],
            NORMALIZE_LIKELIHOOD_BY_DIM,
        )
        logw_old = log_observation_gaussian_unnorm_torch(
            current_paths[-1],
            y_obs_torch,
            obs_biases_torch[ell - 1],
            obs_vars_torch[ell - 1],
            NORMALIZE_LIKELIHOOD_BY_DIM,
        )
        logw = logw_new - logw_old
        ess = ess_from_logweights_torch(logw)

        current_paths, counts, idx = stochastic_round_resample_paths_torch(proposed_paths, logw, seed + 2000 + ell, target_N)

        samples_by_level.append(current_paths[-1])
        paths_by_level.append(current_paths)
        ess_by_level.append(float(ess.detach().cpu()))
        pop_by_level.append(current_paths.shape[1])
        counts_by_level.append(to_np(counts))
        print(f"After level {ell+1}: N={current_paths.shape[1]}, ESS={ess_by_level[-1]:.2f}")

    diagnostics = {
        "ess_by_level": np.asarray(ess_by_level),
        "pop_by_level": np.asarray(pop_by_level),
        "counts_by_level": counts_by_level,
    }

    return current_paths, samples_by_level, paths_by_level, diagnostics


model.eval()
final_paths, samples_by_level, smc_paths_by_level, diagnostics = run_multilevel_smc_observation(
    seed=9876,
    batch_size=BS_SMC,
    target_N=BS_SMC,
)

final_samples = to_np(final_paths[-1])
print("ESS by level:", diagnostics["ess_by_level"])
print("ESS/N by level:", diagnostics["ess_by_level"] / diagnostics["pop_by_level"])
print("Population by level:", diagnostics["pop_by_level"])
print("Final samples:", final_samples.shape)


## 9. W2 diagnostics and semilog rate plots

In [ ]:
# ============================================================
# W2 decay diagnostics against the independent target posterior
# ============================================================

def random_subsample(X, k=None, seed=0):
    X = np.asarray(X)
    if k is None or k >= len(X):
        return X
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=k, replace=False)
    return X[idx]


def sliced_wasserstein(X, Y, num_projections=NUM_PROJECTIONS_W2, max_points=MAX_POINTS_W2, seed=0):
    X = random_subsample(np.asarray(X), max_points, seed=seed)
    Y = random_subsample(np.asarray(Y), max_points, seed=seed + 1)
    k = min(len(X), len(Y))
    X = X[:k]
    Y = Y[:k]
    rng = np.random.default_rng(seed)
    dirs = rng.normal(size=(num_projections, X.shape[1]))
    dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)
    Xp = np.sort(X @ dirs.T, axis=0)
    Yp = np.sort(Y @ dirs.T, axis=0)
    return np.sqrt(np.mean((Xp - Yp) ** 2))


def one_dim_w2(u, v):
    u = np.asarray(u).reshape(-1)
    v = np.asarray(v).reshape(-1)
    k = min(len(u), len(v))
    return np.sqrt(np.mean((np.sort(u[:k]) - np.sort(v[:k])) ** 2))


def residual_w2(X, Y, A, y, max_points=MAX_POINTS_W2, seed=0):
    X = random_subsample(np.asarray(X), max_points, seed=seed)
    Y = random_subsample(np.asarray(Y), max_points, seed=seed + 1)
    RX = X @ A.T - y[None, :]
    RY = Y @ A.T - y[None, :]
    if RX.shape[1] == 1:
        return one_dim_w2(RX[:, 0], RY[:, 0])
    return sliced_wasserstein(RX, RY, num_projections=NUM_PROJECTIONS_W2, max_points=max_points, seed=seed)


def baseline_half(values, levels):
    c = values[0] * 2.0 ** (levels[0] / 2.0)
    return c * 2.0 ** (-levels / 2.0)


sample_sets_np = [to_np(s) for s in samples_by_level]
posterior_ref = np.asarray(posterior_samples)
A_np = np.asarray(A_OBS_np)
y_np = np.asarray(y_obs_np)
levels = np.arange(1, NUM_SMC_LEVELS + 1)

smc_sw2_to_post = np.array([
    sliced_wasserstein(sample_sets_np[ell], posterior_ref, seed=100 + ell)
    for ell in range(NUM_SMC_LEVELS)
])

qn_sw2_to_post = np.array([
    sliced_wasserstein(target_samples_by_level[ell], posterior_ref, seed=200 + ell)
    for ell in range(NUM_SMC_LEVELS)
])

smc_resid_w2_to_post = np.array([
    residual_w2(sample_sets_np[ell], posterior_ref, A_np, y_np, seed=300 + ell)
    for ell in range(NUM_SMC_LEVELS)
])

qn_resid_w2_to_post = np.array([
    residual_w2(target_samples_by_level[ell], posterior_ref, A_np, y_np, seed=400 + ell)
    for ell in range(NUM_SMC_LEVELS)
])

smc_sw2_to_qn = np.array([
    sliced_wasserstein(sample_sets_np[ell], target_samples_by_level[ell], seed=500 + ell)
    for ell in range(NUM_SMC_LEVELS)
])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].semilogy(levels, smc_sw2_to_post, marker="o", label="SMC to posterior")
axes[0].semilogy(levels, qn_sw2_to_post, marker="x", linestyle="--", label=r"$Q_n$ to posterior")
axes[0].semilogy(levels, baseline_half(smc_sw2_to_post, levels), linestyle=":", label=r"SMC baseline $C2^{-n/2}$")
axes[0].set_title(r"Full-dimensional sliced $W_2$ to independent posterior")
axes[0].set_xlabel("level")
axes[0].grid(True, which="both", alpha=0.3)
axes[0].legend()

axes[1].semilogy(levels, smc_resid_w2_to_post, marker="o", label="SMC observation")
axes[1].semilogy(levels, qn_resid_w2_to_post, marker="x", linestyle="--", label=r"$Q_n$ observation")
axes[1].semilogy(levels, baseline_half(smc_resid_w2_to_post, levels), linestyle=":", label=r"SMC baseline $C2^{-n/2}$")
axes[1].set_title(r"Observation-space $W_2$ to independent posterior")
axes[1].set_xlabel("level")
axes[1].grid(True, which="both", alpha=0.3)
axes[1].legend()

axes[2].semilogy(levels, smc_sw2_to_qn, marker="o", label=r"SMC to matching $Q_n$")
axes[2].semilogy(levels, baseline_half(smc_sw2_to_qn, levels), linestyle=":", label=r"baseline $C2^{-n/2}$")
axes[2].set_title(r"SMC approximation error: sliced $W_2$")
axes[2].set_xlabel("level")
axes[2].grid(True, which="both", alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("Sliced W2 uses full DIM-dimensional samples.")
print("Observation W2 uses A_OBS x - y_obs.")
print("Sliced W2 SMC to independent posterior:", smc_sw2_to_post)
print("Sliced W2 Q_n to independent posterior:", qn_sw2_to_post)
print("Observation W2 SMC to independent posterior:", smc_resid_w2_to_post)
print("Observation W2 Q_n to independent posterior:", qn_resid_w2_to_post)
print("Sliced W2 SMC to Q_n:", smc_sw2_to_qn)


In [ ]:
# ============================================================
# Semilog W2 decay plots
# ============================================================

def baseline_half_rate(values, levels, ref_index=0):
    values = np.asarray(values, dtype=float)
    levels = np.asarray(levels, dtype=float)
    n0 = levels[ref_index]
    D0 = values[ref_index]
    return D0 * 2.0 ** (-(levels - n0) / 2.0)


def baseline_one_rate(values, levels, ref_index=0):
    values = np.asarray(values, dtype=float)
    levels = np.asarray(levels, dtype=float)
    n0 = levels[ref_index]
    D0 = values[ref_index]
    return D0 * 2.0 ** (-(levels - n0))


def normalized_half_rate(levels, ref_index=0):
    levels = np.asarray(levels, dtype=float)
    n0 = levels[ref_index]
    return 2.0 ** (-(levels - n0) / 2.0)


def fit_exponent_base2(values, levels, ref_index=0):
    values = np.asarray(values, dtype=float)
    levels = np.asarray(levels, dtype=float)
    D0 = values[ref_index]
    n0 = levels[ref_index]
    ratio = values / D0
    delta_n = levels - n0
    mask = np.isfinite(ratio) & (ratio > 0) & (delta_n > 0)
    if mask.sum() < 1:
        return np.nan
    x = delta_n[mask]
    y = np.log2(ratio[mask])
    slope = np.sum(x * y) / (np.sum(x * x) + 1e-12)
    return -slope


def plot_semilog_decay(values, levels, title, ref_index=0, include_fast_rate=True):
    values = np.asarray(values, dtype=float)
    levels = np.asarray(levels, dtype=float)
    half = baseline_half_rate(values, levels, ref_index=ref_index)
    one = baseline_one_rate(values, levels, ref_index=ref_index)
    a_emp = fit_exponent_base2(values, levels, ref_index=ref_index)

    plt.figure(figsize=(6.8, 4.4))
    plt.semilogy(levels, np.maximum(values, 1e-14), marker="o", linewidth=1.8, label="empirical")
    plt.semilogy(levels, np.maximum(half, 1e-14), linestyle="--", marker="s", linewidth=1.5, label=r"$C2^{-n/2}$")
    if include_fast_rate:
        plt.semilogy(levels, np.maximum(one, 1e-14), linestyle=":", marker="x", linewidth=1.5, label=r"$C2^{-n}$")
    plt.xlabel("level n")
    plt.ylabel(r"$D_n$")
    plt.title(title + rf"  fitted $a={a_emp:.3f}$ in $C2^{{-an}}$")
    plt.xticks(levels)
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_normalized_semilog_decay(values, levels, title, ref_index=0):
    values = np.asarray(values, dtype=float)
    levels = np.asarray(levels, dtype=float)
    D0 = values[ref_index]
    ratio = values / D0
    theory_ratio = normalized_half_rate(levels, ref_index=ref_index)
    a_emp = fit_exponent_base2(values, levels, ref_index=ref_index)

    plt.figure(figsize=(6.8, 4.4))
    plt.semilogy(levels, np.maximum(ratio, 1e-14), marker="o", linewidth=1.8, label=r"empirical $D_n/D_{n_0}$")
    plt.semilogy(levels, np.maximum(theory_ratio, 1e-14), linestyle="--", marker="s", linewidth=1.5, label=r"$2^{-(n-n_0)/2}$")
    plt.xlabel("level n")
    plt.ylabel(r"$D_n/D_{n_0}$")
    plt.title(title + rf"\nnormalized decay, fitted $a={a_emp:.3f}$")
    plt.xticks(levels)
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


series_to_plot = {
    "Full-dimensional sliced W2: SMC to independent posterior": smc_sw2_to_post,
    "Full-dimensional sliced W2: Q_n to independent posterior": qn_sw2_to_post,
    "Observation-space W2: SMC to independent posterior": smc_resid_w2_to_post,
    "Observation-space W2: Q_n to independent posterior": qn_resid_w2_to_post,
    "SMC approximation error: sliced W2 to matching Q_n": smc_sw2_to_qn,
}

for title, values in series_to_plot.items():
    plot_semilog_decay(values, levels, title=title, ref_index=0, include_fast_rate=True)
    plot_normalized_semilog_decay(values, levels, title=title, ref_index=0)


## 10. Scatter comparison

In [ ]:
# ============================================================
# Scatter comparison by level: first two coordinates
# ============================================================

sample_sets_np = [to_np(s) for s in samples_by_level]
posterior_ref = np.asarray(posterior_samples)

all_xy = np.concatenate(
    [posterior_ref[:, :2]]
    + [q[:, :2] for q in target_samples_by_level]
    + [s[:, :2] for s in sample_sets_np],
    axis=0,
)

x_lo, y_lo = np.quantile(all_xy, 0.005, axis=0)
x_hi, y_hi = np.quantile(all_xy, 0.995, axis=0)
pad_x = 0.08 * (x_hi - x_lo + 1e-12)
pad_y = 0.08 * (y_hi - y_lo + 1e-12)

rng = np.random.default_rng(321)
fig, axes = plt.subplots(1, NUM_SMC_LEVELS, figsize=(4.5 * NUM_SMC_LEVELS, 4.2), squeeze=False)
axes = axes[0]

for ell in range(NUM_SMC_LEVELS):
    ax = axes[ell]
    smc = sample_sets_np[ell]
    qn = target_samples_by_level[ell]

    for arr, label, alpha, size in [
        (posterior_ref, "target posterior", 0.18, 8),
        (qn, f"Q_{ell+1}", 0.25, 8),
        (smc, "SMC", 0.55, 8),
    ]:
        idx = rng.choice(len(arr), size=min(2000, len(arr)), replace=False)
        ax.scatter(arr[idx, 0], arr[idx, 1], s=size, alpha=alpha, label=label)

    ax.scatter([float(mu_final_np[0])], [float(mu_final_np[1])], c="red", marker="x", s=120, linewidths=2, label="Gaussian target" if ell == 0 else None)
    ax.set_xlim(x_lo - pad_x, x_hi + pad_x)
    ax.set_ylim(y_lo - pad_y, y_hi + pad_y)
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")
    ax.set_title(f"level {ell+1}")
    ax.grid(True, alpha=0.3)
    if ell == 0:
        ax.legend(fontsize=8)

plt.suptitle("SMC vs level target Q_n vs independent posterior (x0, x1)")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Scatter comparison by level: selected coordinate triples
# Default: PLOT_COORDS = [0, 1, 2]
# ============================================================

def w2_1d(x, y):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    k = min(len(x), len(y))
    return np.sqrt(np.mean((np.sort(x)[:k] - np.sort(y)[:k]) ** 2))


posterior_plot = np.asarray(posterior_samples)
sample_sets_plot = [to_np(s) for s in samples_by_level]
target_samples_plot = [np.asarray(s) for s in target_samples_by_level]
mu_np = mu_final_np.reshape(-1)

coords = list(PLOT_COORDS)
assert len(coords) == 3, "PLOT_COORDS must have exactly three entries."
assert max(coords) < DIM, f"PLOT_COORDS={coords} out of range for DIM={DIM}."
coord_pairs = [(coords[0], coords[1]), (coords[0], coords[2]), (coords[1], coords[2])]

print("Coordinates plotted:", coords)
print("Coordinate pairs:", coord_pairs)
print("Posterior coordinate mean:", posterior_plot[:, coords].mean(axis=0))
print("Posterior coordinate std:", posterior_plot[:, coords].std(axis=0))
print("Target point coordinates:", mu_np[coords])

all_arrays = [posterior_plot] + target_samples_plot + sample_sets_plot
all_coord_values = np.concatenate([arr[:, coords] for arr in all_arrays], axis=0)

range_by_coord = {}
for local_j, coord in enumerate(coords):
    lo, hi = np.quantile(all_coord_values[:, local_j], [0.005, 0.995])
    pad = 0.08 * (hi - lo + 1e-12)
    range_by_coord[coord] = (lo - pad, hi + pad)

print("\nMarginal W2 to independent posterior on selected coordinates:")
for ell in range(NUM_SMC_LEVELS):
    smc = sample_sets_plot[ell]
    qn = target_samples_plot[ell]
    smc_w2 = [w2_1d(smc[:, c], posterior_plot[:, c]) for c in coords]
    qn_w2 = [w2_1d(qn[:, c], posterior_plot[:, c]) for c in coords]
    print(f"level {ell+1}: SMC {smc_w2}; Q_n {qn_w2}")

rng = np.random.default_rng(321)
fig, axes = plt.subplots(len(coord_pairs), NUM_SMC_LEVELS, figsize=(4.5 * NUM_SMC_LEVELS, 4.2 * len(coord_pairs)), squeeze=False)

for row, (i, j) in enumerate(coord_pairs):
    for ell in range(NUM_SMC_LEVELS):
        ax = axes[row, ell]
        smc = sample_sets_plot[ell]
        qn = target_samples_plot[ell]

        for arr, label, alpha, size in [
            (posterior_plot, "target posterior", 0.18, 8),
            (qn, f"Q_{ell+1}", 0.25, 8),
            (smc, "SMC", 0.55, 8),
        ]:
            idx = rng.choice(len(arr), size=min(2000, len(arr)), replace=False)
            ax.scatter(arr[idx, i], arr[idx, j], s=size, alpha=alpha, label=label)

        ax.scatter([float(mu_np[i])], [float(mu_np[j])], c="red", marker="x", s=120, linewidths=2, label="Gaussian target" if (ell == 0 and row == 0) else None)
        ax.set_xlim(range_by_coord[i])
        ax.set_ylim(range_by_coord[j])
        ax.set_xlabel(f"x{i}")
        ax.set_ylabel(f"x{j}")
        ax.set_title(f"level {ell+1}: x{i} vs x{j}")
        ax.grid(True, alpha=0.3)

        if ell == 0 and row == 0:
            ax.legend(fontsize=8)

plt.suptitle("SMC vs level target Q_n vs independent posterior\n" f"coordinate pairwise projections: {coord_pairs}", y=1.02)
plt.tight_layout()
plt.show()
